# Advanced Training and Production Inference

<div style="display: flex; gap: 10px; margin-bottom: 20px;">
  <a href="https://colab.research.google.com/github/basf/DeepTab/blob/main/docs/tutorials/notebooks/advanced_training.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
  <a href="https://github.com/basf/DeepTab/blob/main/docs/tutorials/notebooks/advanced_training.ipynb" target="_blank">
    <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github&logoColor=white" alt="View on GitHub"/>
  </a>
</div>

This end-to-end tutorial covers three topics that come up after the basics:
customising the optimizer and scheduler, extending the built-in registries with
your own implementations, and deploying a trained model with `InferenceModel`.

**What You Will Learn**

- How to discover available optimizers and schedulers at runtime.
- How to pass `optimizer_type`, `optimizer_kwargs`, and scheduler fields through `TrainerConfig`.
- What `no_weight_decay_for_bias_and_norm` does and when to use it.
- How to register a custom optimizer or scheduler.
- How to use `InferenceModel` for schema-validated, deployment-friendly inference.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from deeptab import InferenceModel
from deeptab.configs import MambularConfig, PreprocessingConfig, TrainerConfig
from deeptab.models import MambularClassifier
from deeptab.training import (
    available_optimizers,
    available_schedulers,
    register_optimizer,
    register_scheduler,
)

## Data

In [ ]:
RANDOM_STATE = 42

X_num, y = make_classification(
    n_samples=1500,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    random_state=RANDOM_STATE,
)

X = pd.DataFrame(X_num, columns=[f"feat_{i}" for i in range(X_num.shape[1])])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE
)

---

## Part 1 — Optimizers

### Discovering available optimizers

In [ ]:
print(available_optimizers())

### Using AdamW with custom kwargs

In [ ]:
trainer = TrainerConfig(
    max_epochs=40,
    batch_size=128,
    lr=3e-4,
    patience=10,
    optimizer_type="AdamW",
    optimizer_kwargs={
        "betas": (0.9, 0.98),
        "eps": 1e-8,
    },
    weight_decay=1e-2,
)

clf = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=trainer,
    random_state=RANDOM_STATE,
)
clf.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("AdamW AUROC:", roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]))

### Weight-decay exemption for bias and normalisation layers

`no_weight_decay_for_bias_and_norm=True` splits the model parameters into two groups:
one with `weight_decay` as configured and one (biases and normalisation weights) with
`weight_decay=0`. This is the recommended practice for transformer-style architectures.

In [ ]:
clf_wd = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=40,
        batch_size=128,
        lr=3e-4,
        patience=10,
        optimizer_type="AdamW",
        weight_decay=1e-2,
        no_weight_decay_for_bias_and_norm=True,
    ),
    random_state=RANDOM_STATE,
)
clf_wd.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("AdamW + no-WD-BN AUROC:", roc_auc_score(y_test, clf_wd.predict_proba(X_test)[:, 1]))

---

## Part 2 — Schedulers

### Discovering available schedulers

In [ ]:
print(available_schedulers())

### CosineAnnealingLR

In [ ]:
clf_cos = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=60,
        batch_size=128,
        lr=3e-4,
        patience=12,
        optimizer_type="AdamW",
        weight_decay=1e-2,
        scheduler_type="CosineAnnealingLR",
        scheduler_kwargs={"T_max": 60, "eta_min": 1e-6},
        scheduler_interval="epoch",
    ),
    random_state=RANDOM_STATE,
)
clf_cos.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("CosineAnnealingLR AUROC:", roc_auc_score(y_test, clf_cos.predict_proba(X_test)[:, 1]))

### ReduceLROnPlateau (default)

In [ ]:
clf_plateau = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=60,
        batch_size=128,
        lr=3e-4,
        patience=12,
        optimizer_type="AdamW",
        weight_decay=1e-2,
        scheduler_type="ReduceLROnPlateau",
        scheduler_monitor="val_loss",
        scheduler_kwargs={
            "factor": 0.5,
            "patience": 5,
            "min_lr": 1e-6,
        },
    ),
    random_state=RANDOM_STATE,
)
clf_plateau.fit(X_train, y_train, X_val=X_val, y_val=y_val)

### Disabling the scheduler

In [ ]:
clf_const_lr = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=40,
        batch_size=128,
        lr=3e-4,
        patience=10,
        scheduler_type=None,
    ),
    random_state=RANDOM_STATE,
)
clf_const_lr.fit(X_train, y_train, X_val=X_val, y_val=y_val)

---

## Part 3 — Custom Optimizer and Scheduler Registration

### Registering a custom optimizer

In [ ]:
class ScaledAdam(torch.optim.Adam):
    """Adam with gradient pre-scaling (toy example)."""

    def __init__(self, params, lr=1e-3, scale=1.0, **kwargs):
        super().__init__(params, lr=lr * scale, **kwargs)


register_optimizer("ScaledAdam", ScaledAdam)
print("ScaledAdam registered:", "ScaledAdam" in available_optimizers())

clf_custom_opt = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=30,
        batch_size=128,
        lr=3e-4,
        patience=8,
        optimizer_type="ScaledAdam",
        optimizer_kwargs={"scale": 0.8},
    ),
    random_state=RANDOM_STATE,
)
clf_custom_opt.fit(X_train, y_train, X_val=X_val, y_val=y_val)

### Registering a custom scheduler

In [ ]:
class WarmupConstant(torch.optim.lr_scheduler.LambdaLR):
    """Linear warmup for `warmup_steps`, then constant LR."""

    def __init__(self, optimizer, warmup_steps: int = 100):
        def _lambda(step: int) -> float:
            if step < warmup_steps:
                return float(step) / max(1, warmup_steps)
            return 1.0

        super().__init__(optimizer, lr_lambda=_lambda)


register_scheduler("WarmupConstant", WarmupConstant)
print("WarmupConstant registered:", "WarmupConstant" in available_schedulers())

clf_warmup = MambularClassifier(
    model_config=MambularConfig(d_model=64, n_layers=3),
    preprocessing_config=PreprocessingConfig(numerical_preprocessing="quantile"),
    trainer_config=TrainerConfig(
        max_epochs=40,
        batch_size=128,
        lr=3e-4,
        patience=10,
        scheduler_type="WarmupConstant",
        scheduler_kwargs={"warmup_steps": 200},
        scheduler_interval="step",
    ),
    random_state=RANDOM_STATE,
)
clf_warmup.fit(X_train, y_train, X_val=X_val, y_val=y_val)

---

## Part 4 — Production Inference with `InferenceModel`

`InferenceModel` wraps a fitted estimator and exposes only the prediction
surface. Training methods (`fit`, `optimize_hparams`, etc.) are absent.

### Save a model to disk

In [ ]:
clf_wd.save("advanced_clf.pt")

### Load via `from_path`

In [ ]:
model = InferenceModel.from_path("advanced_clf.pt")
print(model)
print("Task:", model.task)
print("Features:", model.n_features)

### Wrap an already-fitted estimator

In [ ]:
model_live = InferenceModel.from_estimator(clf_wd)
print(model_live.task)
print(model_live.n_features)

### Introspection

In [ ]:
info = model.describe()
print(list(info.keys()))

rt = model.runtime_info()
print(list(rt.keys()))

### Schema validation

In [ ]:
# Happy path
X_clean = model.validate_input(X_test)
print("Schema valid, shape:", X_clean.shape)

# Missing column
X_bad = X_test.drop(columns=["feat_0"])
try:
    model.validate_input(X_bad)
except ValueError as exc:
    print("Missing column error:", exc)

# Extra columns — dropped with a warning
X_wide = X_test.copy()
X_wide["audit_id"] = range(len(X_test))
X_clean = model.validate_input(X_wide, allow_extra_columns=True)
print("After dropping extra column, shape:", X_clean.shape)

### Prediction

In [ ]:
# Hard class labels
labels = model.predict(X_clean)
print("Accuracy:", accuracy_score(y_test, labels))

# Class probabilities
proba = model.predict_proba(X_clean)
print("AUROC:", roc_auc_score(y_test, proba[:, 1]))

### Production service pattern

A minimal service handler using `InferenceModel`:

In [ ]:
# Module-level: load once at startup
_MODEL = InferenceModel.from_path("advanced_clf.pt")


def score(payload: dict) -> dict:
    X = pd.DataFrame([payload])
    X_clean = _MODEL.validate_input(X, allow_extra_columns=True)
    proba   = _MODEL.predict_proba(X_clean)
    label   = _MODEL.predict(X_clean)
    return {
        "probability_positive": float(proba[0, 1]),
        "label": int(label[0]),
    }


# Example call
sample = X_test.iloc[0].to_dict()
result = score(sample)
print(result)

---

## Next Steps

- [Core concepts: training and evaluation](../../core_concepts/training_and_evaluation)
- [Core concepts: inference](../../core_concepts/inference)
- [Imbalanced classification tutorial](imbalance_classification)
- [Regression tutorial](regression)